# Daily Climate Data for WxChallenge
Download daily temperature and precipitation observations from NOAA NCEI's Climate Data Online (CDO) API for September 29–October 9, 2026 (inclusive).

**Note:** This uses the `GHCND` daily summaries dataset.

In [26]:
import requests
import pandas as pd
import datetime as dt

import yaml
from pathlib import Path
config_path = "config/config.yml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
TOKEN = config['ncei']['token']

# Identify stations we need. 
1) Week one used KTPA (Tampa International Airport) as the station.
2) Week two used KOLM (Olympia Regional Airport) as the station. 

In [34]:
# We use this to identify stations we need. 
GHCN_station_list_url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"

# read the station list into a DataFrame
stations_df = pd.read_fwf(GHCN_station_list_url, header=None, widths=[11, 9, 10, 7, 3, 31, 4, 4], names=["stationid", "latitude", "longitude", "elevation", "state", "name", "gsn_flag", "hcn_crn_flag"])
stations_df

,stationid,latitude,longitude,elevation,state,name,gsn_flag,hcn_crn_flag
0,ACW00011604,17.1167,-61.7833,10.1,NaN,ST JOHNS COOLIDGE FLD,NaN,NaN
1,ACW00011647,17.1333,-61.7833,19.2,NaN,ST JOHNS,NaN,NaN
2,AE000041196,25.3330,55.5170,34.0,NaN,SHARJAH INTER. AIRP,GSN,NaN
3,AEM00041194,25.2550,55.3640,10.4,NaN,DUBAI INTL,NaN,NaN
4,AEM00041217,24.4330,54.6510,26.8,NaN,ABU DHABI INTL,NaN,NaN
...,...,...,...,...,...,...,...,...
133034,ZI000067969,-21.0500,29.3670,861.0,NaN,WEST NICHOLSON,NaN,NaN
133035,ZI000067975,-20.0670,30.8670,1095.0,NaN,MASVINGO,NaN,NaN
133036,ZI000067977,-21.0170,31.5830,430.0,NaN,BUFFALO RANGE,NaN,NaN
133037,ZI000067983,-20.2000,32.6160,1132.0,NaN,CHIPINGE,GSN,NaN


In [35]:
# We can filter the DataFrame to find stations in Washington (WA) that have "OLY" in their name, which likely refers to Olympia. 
# Here's how you can do that:
stations_df[(stations_df['state'] == 'WA') & (stations_df['name'].str.contains(r'OLY.*AP', case=False, regex=True, na=False))]

,stationid,latitude,longitude,elevation,state,name,gsn_flag,hcn_crn_flag
131407,USW00024227,46.9736,-122.905,61.0,WA,OLYMPIA AP,GSN,NaN


In [21]:
# set IDs for KTPA and KOLM stations
ktpa_id = stations_df[(stations_df['state'] == 'FL') & (stations_df['name'].str.contains(r'TAMPA.*INTL', case=False, regex=True, na=False))]['stationid'].values[0]
kolm_id = stations_df[(stations_df['state'] == 'WA') & (stations_df['name'].str.contains(r'OLY.*AP', case=False, regex=True, na=False))]['stationid'].values[0]
print(ktpa_id)
print(kolm_id)

USW00012842
USW00024227


# Function to Fetch NCEI Data
For more informaiton visit [NCEI](https://www.ncei.noaa.gov/support/access-data-service-api-user-documentation) Documentation.

Be sure to request a toke and put it in your [config.yml](/config/config.yml)

In [86]:
TOKEN = config['ncei']['token']

STATION = f"GHCND:{kolm_id}"  # Tampa International Airport (KTPA)
START_DATE = "2026-09-29"
END_DATE = "2026-10-09"

In [87]:
# Show the available datatypes for the GHCND dataset and the specified station.
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/datatypes"

params = {
    "datasetid": "GHCND",
    "stationid": STATION,
    "limit": 1000
}

response = requests.get(
    url,
    headers={"token": TOKEN},
    params=params,
    timeout=30
)
response.raise_for_status()

datatypes = pd.DataFrame(response.json()["results"])

display(datatypes[["id", "name"]])

,id,name
0,ACMH,Average cloudiness midnight to midnight from m...
1,ACSH,Average cloudiness sunrise to sunset from manu...
2,AWND,Average wind speed
3,FMTM,Time of fastest mile or fastest 1-minute wind
4,FRGT,Top of frozen ground layer
5,PGTM,Peak gust time
6,PRCP,Precipitation
7,SNOW,Snowfall
8,SNWD,Snow depth
9,TAVG,Average Temperature.


In [88]:
def fetch_params(station_id, start_date, end_date):
    """
    Fetches weather parameters from the NOAA API for a specific station, date range, and datatype.
    
    Args:
        station_id (str): The station ID in the format "GHCND:XXXX".
        start_date (str): The start date for the data range.
        end_date (str): The end date for the data range.
        datatype_id (str): The ID of the datatype to fetch.

    """
    if station_id is None or start_date is None or end_date is None:
        raise ValueError("station_id, start_date, and end_date must be provided")
    if station_id.startswith("GHCND:") == False:
        raise ValueError("station_id must start with 'GHCND:'")
    
    url = "https://www.ncei.noaa.gov/cdo-web/api/v2/datatypes"
    params = {
        "datasetid": "GHCND",
        "stationid": STATION,
        "limit": 1000
    }
    response = requests.get(
        url,
        headers={"token": TOKEN},
        params=params,
        timeout=30
    )
    response.raise_for_status()
    datatypes = pd.DataFrame(response.json()["results"])
    display(datatypes[["id", "name"]])



In [89]:
fetch_params(STATION, START_DATE, END_DATE)

,id,name
0,ACMH,Average cloudiness midnight to midnight from m...
1,ACSH,Average cloudiness sunrise to sunset from manu...
2,AWND,Average wind speed
3,FMTM,Time of fastest mile or fastest 1-minute wind
4,FRGT,Top of frozen ground layer
5,PGTM,Peak gust time
6,PRCP,Precipitation
7,SNOW,Snowfall
8,SNWD,Snow depth
9,TAVG,Average Temperature.


In [90]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
params = {
    "datasetid": "GHCND",
    "stationid": STATION,
    "startdate": START_DATE,
    "enddate": END_DATE,
    "datatypeid": "TMAX,TMIN,WSF2,PRCP",
    "units": "standard",
    "limit": 1000,
}

if TOKEN == "PASTE_YOUR_NCEI_TOKEN_HERE":
    raise ValueError("Paste your NCEI token into the TOKEN variable first.")

response = requests.get(url, headers={"token": TOKEN}, params=params, timeout=30)
response.raise_for_status()
records = response.json().get("results", [])
print(f"Downloaded {len(records)} records.")

Downloaded 44 records.


In [91]:
if not records:
    print("No records returned. Check station availability, dates, and NCEI processing delays.")
    df = pd.DataFrame()
else:
    raw = pd.DataFrame(records)
    df = raw.pivot_table(index="date", columns="datatype", values="value", aggfunc="first")
    df.index = pd.to_datetime(df.index).date
    df.index.name = "date"
    df = df.sort_index()
    df['Wind (knots)'] = round(df['WSF2'] / 1.151, 1)  # Convert mph to knots
    display(df)

# Standard units: TMAX/TMIN in degrees F; PRCP in inches.

datatype,PRCP,TMAX,TMIN,WSF2,Wind (knots)
date,,,,,
2026-09-29,0.22,61.0,51.0,17.0,14.8
2026-09-30,0.00,66.0,45.0,13.0,11.3
2026-10-01,0.00,71.0,43.0,6.9,6.0
2026-10-02,0.00,73.0,46.0,12.1,10.5
2026-10-03,0.00,70.0,46.0,14.1,12.3
2026-10-04,0.00,72.0,43.0,8.1,7.0
2026-10-05,0.00,70.0,46.0,6.0,5.2
2026-10-06,0.00,68.0,50.0,12.1,10.5
2026-10-07,0.00,68.0,47.0,16.1,14.0


In [104]:
def fetch_wx_data(STATION, START_DATE, END_DATE, DATATYPE_ID="TMAX,TMIN,WSF2,PRCP"):
    url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
    params = {
        "datasetid": "GHCND",
        "stationid": STATION,
        "startdate": START_DATE,
        "enddate": END_DATE,
        "datatypeid": DATATYPE_ID,
        "units": "standard",
        "limit": 1000,
    }

    if TOKEN == "PASTE_YOUR_NCEI_TOKEN_HERE":
        raise ValueError("Paste your NCEI token into the TOKEN variable first.")

    response = requests.get(url, headers={"token": TOKEN}, params=params, timeout=30)
    response.raise_for_status()
    records = response.json().get("results", [])
    print(f"Downloaded {len(records)} records.")

    if not records:
        print("No records returned. Check station availability, dates, and NCEI processing delays.")
        df = pd.DataFrame()
    else:
        raw = pd.DataFrame(records)
        df = raw.pivot_table(index="date", columns="datatype", values="value", aggfunc="first")
        df.index = pd.to_datetime(df.index).date
        df.index.name = "date"
        df = df.sort_index()
        df['Wind (knots)'] = round(df['WSF2'] / 1.151, 1)  # Convert mph to knots
        df = df[['TMAX', 'TMIN', 'Wind (knots)', 'PRCP']]  # Reorder columns
        df = df.rename(columns={'TMAX': 'High', 'TMIN': 'Low', 'Wind (knots)': 'Wind', 'PRCP': 'Precip'})  # Rename columns to match WXChallenge for clarity
        display(df)
        return df

# Test
test_df = fetch_wx_data(STATION, START_DATE, END_DATE)

Downloaded 44 records.


datatype,High,Low,Wind,Precip
date,,,,
2026-09-29,61.0,51.0,14.8,0.22
2026-09-30,66.0,45.0,11.3,0.00
2026-10-01,71.0,43.0,6.0,0.00
2026-10-02,73.0,46.0,10.5,0.00
2026-10-03,70.0,46.0,12.3,0.00
2026-10-04,72.0,43.0,7.0,0.00
2026-10-05,70.0,46.0,5.2,0.00
2026-10-06,68.0,50.0,10.5,0.00
2026-10-07,68.0,47.0,14.0,0.00


In [105]:
test_df

datatype,High,Low,Wind,Precip
date,,,,
2026-09-29,61.0,51.0,14.8,0.22
2026-09-30,66.0,45.0,11.3,0.00
2026-10-01,71.0,43.0,6.0,0.00
2026-10-02,73.0,46.0,10.5,0.00
2026-10-03,70.0,46.0,12.3,0.00
2026-10-04,72.0,43.0,7.0,0.00
2026-10-05,70.0,46.0,5.2,0.00
2026-10-06,68.0,50.0,10.5,0.00
2026-10-07,68.0,47.0,14.0,0.00


# Putting it all togetether

In [85]:
wx_challenge_dict = {
    "ktpa": {
        "station_id": f'GHCND:{ktpa_id}',
        "start_date": "2026-09-29",
        "end_date": "2026-10-12",
        "dataframe": None
    },
    "kolm": {
        "station_id": f'GHCND:{kolm_id}',
        "start_date": "2026-10-10",
        "end_date": "2026-10-26",
        "dataframe": None
    }
}

ktpa = fetch_wx_data(wx_challenge_dict["ktpa"]["station_id"], wx_challenge_dict["ktpa"]["start_date"], wx_challenge_dict["ktpa"]["end_date"])
kolm = fetch_wx_data(wx_challenge_dict["kolm"]["station_id"], wx_challenge_dict["kolm"]["start_date"], wx_challenge_dict["kolm"]["end_date"])

Downloaded 44 records.


datatype,High,Low,Wind,Precip
date,,,,
2026-09-29,87.0,74.0,10.5,0.00
2026-09-30,88.0,75.0,14.0,0.42
2026-10-01,88.0,74.0,14.0,0.28
2026-10-02,87.0,76.0,10.5,0.00
2026-10-03,90.0,80.0,11.3,0.00
2026-10-04,89.0,79.0,8.8,0.01
2026-10-05,88.0,78.0,11.3,0.00
2026-10-06,79.0,76.0,7.7,1.36
2026-10-07,83.0,73.0,22.5,2.16


Downloaded 0 records.
No records returned. Check station availability, dates, and NCEI processing delays.


In [94]:
wx_challenge_dict["kolm"]

{'station_id': 'GHCND:USW00024227',
 'start_date': '2026-10-10',
 'end_date': '2026-10-26',
 'dataframe': None}

In [99]:
fetch_wx_data('GHCND:USW00024227', '2026-10-02', '2026-10-13')

Downloaded 32 records.


datatype,High,Low,Wind,Precip
date,,,,
2026-10-02,73.0,46.0,10.5,0.0
2026-10-03,70.0,46.0,12.3,0.0
2026-10-04,72.0,43.0,7.0,0.0
2026-10-05,70.0,46.0,5.2,0.0
2026-10-06,68.0,50.0,10.5,0.0
2026-10-07,68.0,47.0,14.0,0.0
2026-10-08,73.0,43.0,4.3,0.0
2026-10-09,62.0,40.0,19.0,0.0


In [102]:
response.raise_for_status()
records = response.json().get("results", [])

raw = pd.DataFrame(records)

if not raw.empty:
    display(
        raw.groupby(["date", "datatype"])
           .size()
           .unstack(fill_value=0)
    )

datatype,PRCP,TMAX,TMIN,WSF2
date,,,,
2026-09-29T00:00:00,1,1,1,1
2026-09-30T00:00:00,1,1,1,1
2026-10-01T00:00:00,1,1,1,1
2026-10-02T00:00:00,1,1,1,1
2026-10-03T00:00:00,1,1,1,1
2026-10-04T00:00:00,1,1,1,1
2026-10-05T00:00:00,1,1,1,1
2026-10-06T00:00:00,1,1,1,1
2026-10-07T00:00:00,1,1,1,1


In [101]:
response.json()

{'metadata': {'resultset': {'offset': 1, 'count': 44, 'limit': 1000}},
 'results': [{'date': '2026-09-29T00:00:00',
   'datatype': 'PRCP',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,2400',
   'value': 0.22},
  {'date': '2026-09-29T00:00:00',
   'datatype': 'TMAX',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,2400',
   'value': 61.0},
  {'date': '2026-09-29T00:00:00',
   'datatype': 'TMIN',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,2400',
   'value': 51.0},
  {'date': '2026-09-29T00:00:00',
   'datatype': 'WSF2',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,',
   'value': 17.0},
  {'date': '2026-09-30T00:00:00',
   'datatype': 'PRCP',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,2400',
   'value': 0.0},
  {'date': '2026-09-30T00:00:00',
   'datatype': 'TMAX',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,2400',
   'value': 66.0},
  {'date': '2026-09-30T00:00:00',
   'datatype': 'TMIN',
   'station': 'GHCND:USW00

## Need to test a different function using the Local Climate Data version 2 dataset
The data from WxChallenge is actually not Midnight to Midnight and the API endpoints used above are not 'live'data. Need to find a different source to pull live metar data. 

# Going back to IEM API

In [136]:
import pandas as pd
import requests
from io import StringIO

def fetch_metar_data(STATION, START_DATE, END_DATE):

    url = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"

    # IEM generally uses the 3-letter station identifier
    station = STATION.removeprefix("K")

    params = {
        "station": station,
        "data": ["tmpf", "dwpf", "sknt", "gust", "p01i"],
        "year1": pd.Timestamp(START_DATE).year,
        "month1": pd.Timestamp(START_DATE).month,
        "day1": pd.Timestamp(START_DATE).day,
        "year2": pd.Timestamp(END_DATE).year,
        "month2": pd.Timestamp(END_DATE).month,
        "day2": pd.Timestamp(END_DATE).day,
        "tz": "Etc/UTC",
        "format": "onlycomma",
        "missing": "empty",
        "report_type": [1, 3, 4],
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    df = pd.read_csv(
        StringIO(response.text),
        comment="#",
        na_values=["M", ""]
    )

    df["valid"] = pd.to_datetime(df["valid"], utc=True)

    for col in ["tmpf", "dwpf", "sknt", "gust", "p01i"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values("valid").reset_index(drop=True)

    print(f"Retrieved {len(df)} observations for {STATION}")

    return df

In [154]:
response.json()

{'metadata': {'resultset': {'offset': 1, 'count': 44, 'limit': 1000}},
 'results': [{'date': '2026-09-29T00:00:00',
   'datatype': 'PRCP',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,2400',
   'value': 0.22},
  {'date': '2026-09-29T00:00:00',
   'datatype': 'TMAX',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,2400',
   'value': 61.0},
  {'date': '2026-09-29T00:00:00',
   'datatype': 'TMIN',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,2400',
   'value': 51.0},
  {'date': '2026-09-29T00:00:00',
   'datatype': 'WSF2',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,',
   'value': 17.0},
  {'date': '2026-09-30T00:00:00',
   'datatype': 'PRCP',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,2400',
   'value': 0.0},
  {'date': '2026-09-30T00:00:00',
   'datatype': 'TMAX',
   'station': 'GHCND:USW00024227',
   'attributes': ',,1,2400',
   'value': 66.0},
  {'date': '2026-09-30T00:00:00',
   'datatype': 'TMIN',
   'station': 'GHCND:USW00

In [142]:
metar = fetch_metar_data(
    "KTPA",
    "2026-09-29",
    "2026-10-10"
)

display(metar.tail(20))

Retrieved 3512 observations for KTPA


,station,valid,tmpf,dwpf,sknt,gust,p01i
3492,TPA,2026-10-09 22:30:00+00:00,NaN,NaN,8.0,NaN,NaN
3493,TPA,2026-10-09 22:35:00+00:00,NaN,NaN,6.0,NaN,NaN
3494,TPA,2026-10-09 22:40:00+00:00,NaN,NaN,7.0,NaN,NaN
3495,TPA,2026-10-09 22:45:00+00:00,NaN,NaN,7.0,NaN,NaN
3496,TPA,2026-10-09 22:50:00+00:00,NaN,NaN,6.0,NaN,NaN
3497,TPA,2026-10-09 22:53:00+00:00,85.0,73.0,8.0,NaN,0.0
3498,TPA,2026-10-09 22:55:00+00:00,NaN,NaN,6.0,NaN,NaN
3499,TPA,2026-10-09 23:00:00+00:00,NaN,NaN,6.0,NaN,NaN
3500,TPA,2026-10-09 23:05:00+00:00,NaN,NaN,7.0,NaN,NaN
3501,TPA,2026-10-09 23:10:00+00:00,NaN,NaN,6.0,NaN,NaN


In [152]:
metar["date"] = metar["valid"].dt.date + dt.timedelta(hours=6)  # Adjusting for UTC to local time (UTC-6 for Tampa, plus 3 hours for the time of day adjustment)

daily_wind = (
    metar
    .groupby("date")
    .agg(
        max_wind_kt=("sknt", "max"),
        max_gust_kt=("gust", "max"),
        observations=("sknt", "count")
    )
)

display(daily_wind)

,max_wind_kt,max_gust_kt,observations
date,,,
2026-09-29,9.0,NaN,308
2026-09-30,11.0,17.0,312
2026-10-01,14.0,23.0,314
2026-10-02,9.0,17.0,305
2026-10-03,11.0,17.0,316
2026-10-04,10.0,16.0,311
2026-10-05,10.0,20.0,314
2026-10-06,9.0,16.0,325
2026-10-07,10.0,NaN,321


# Other stuff below here...

In [32]:
# create a list of specific dates that are used in the WX challenge.
# Tuesday, Wednesday, Thursday, Friday of each weak. 
start_date = dt.date(2026, 9, 29)  # Example start date
end_date = dt.date(2026, 10, 31)   # Example end date

# 1 = Tuesday, 2 = Wednesday, 3 = Thursday, 4 = Friday
# In Python's date.weekday(): Monday is 0, Tuesday is 1, ..., Sunday is 6
tue_to_fri_dates = [
    start_date + dt.timedelta(days=i)
    for i in range((end_date - start_date).days + 1)
    if 1 <= (start_date + dt.timedelta(days=i)).weekday() <= 4
]

print(tue_to_fri_dates)

[datetime.date(2026, 9, 29), datetime.date(2026, 9, 30), datetime.date(2026, 10, 1), datetime.date(2026, 10, 2), datetime.date(2026, 10, 6), datetime.date(2026, 10, 7), datetime.date(2026, 10, 8), datetime.date(2026, 10, 9), datetime.date(2026, 10, 13), datetime.date(2026, 10, 14), datetime.date(2026, 10, 15), datetime.date(2026, 10, 16), datetime.date(2026, 10, 20), datetime.date(2026, 10, 21), datetime.date(2026, 10, 22), datetime.date(2026, 10, 23), datetime.date(2026, 10, 27), datetime.date(2026, 10, 28), datetime.date(2026, 10, 29), datetime.date(2026, 10, 30)]
